# NB-03: Torneio de Estratégias Matemáticas

**Objetivo**: Testar 9+ sistemas matemáticos de apostas sobre os 3.9M rounds e ranquear por performance.

## Estratégias testadas:
1. **Martingale** (baseline atual)
2. **Anti-Martingale (Paroli)** - dobra após ganho
3. **Fibonacci** - sequência de Fibonacci nas perdas
4. **D'Alembert** - +1 após perda, -1 após ganho
5. **Labouchère** - sistema de lista
6. **Oscar's Grind** - aumenta 1 unidade após ganho
7. **Kelly Criterion** - aposta ótima baseada em edge
8. **Fixed Fraction** - sempre X% da banca
9. **Trigger Adaptativo** - trigger dinâmico baseado em volatilidade recente

In [ ]:
import sys
sys.path.insert(0, '..')

# Carregar framework do backtester
%run 12_backtester.ipynb

In [ ]:
# Carregar dados
df = load_raw_data()
multipliers = df['multiplicador'].values
print(f"Dados: {len(multipliers):,} rounds")

## 1. Implementação das Estratégias

In [ ]:
class AntiMartingaleStrategy(BettingStrategy):
    """Paroli: dobra após WIN, reseta após LOSS.
    Lógica inversa do Martingale - captura sequências de vitórias."""

    def __init__(self, trigger=6, target=2.0, max_wins=3):
        self.trigger = trigger
        self.target = target
        self.max_wins = max_wins  # Reseta após N wins seguidos
        self.reset()

    @property
    def name(self):
        return f"Anti-Martingale (T{self.trigger}, max{self.max_wins}w)"

    def reset(self):
        self.consecutive_lows = 0
        self.triggered = False
        self.win_streak = 0
        self.current_mult = 1

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.triggered:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.trigger:
                self.triggered = True
                self.current_mult = 1
                self.win_streak = 0
                return BetDecision(True, base_bet, self.target)
            return BetDecision(False)

        if multiplier >= self.target:  # WIN
            self.win_streak += 1
            if self.win_streak >= self.max_wins:
                self.triggered = False
                self.consecutive_lows = 0
                return BetDecision(False)
            self.current_mult *= 2
            return BetDecision(True, base_bet * self.current_mult, self.target)
        else:  # LOSS
            self.triggered = False
            self.consecutive_lows = 0 if not is_low else 1
            return BetDecision(False)

In [ ]:
class FibonacciStrategy(BettingStrategy):
    """Fibonacci: aposta segue sequência de Fibonacci nas perdas.
    Após perda: avança na sequência. Após ganho: recua 2 posições."""

    def __init__(self, trigger=6, target=2.0, max_level=8):
        self.trigger = trigger
        self.target = target
        self.max_level = max_level
        self.fib = self._generate_fib(max_level + 2)
        self.reset()

    def _generate_fib(self, n):
        seq = [1, 1]
        for _ in range(n - 2):
            seq.append(seq[-1] + seq[-2])
        return seq

    @property
    def name(self):
        return f"Fibonacci (T{self.trigger}, max{self.max_level})"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.fib_level = 0

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.trigger:
                self.in_sequence = True
                self.fib_level = 0
                return BetDecision(True, base_bet * self.fib[0], self.target)
            return BetDecision(False)

        if multiplier >= self.target:  # WIN
            self.fib_level = max(0, self.fib_level - 2)
            if self.fib_level == 0:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            return BetDecision(True, base_bet * self.fib[self.fib_level], self.target)
        else:  # LOSS
            self.fib_level += 1
            if self.fib_level >= self.max_level:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            return BetDecision(True, base_bet * self.fib[self.fib_level], self.target)

In [ ]:
class DAlembertStrategy(BettingStrategy):
    """D'Alembert: +1 unidade após perda, -1 após ganho.
    Progressão linear, mais conservador que Martingale."""

    def __init__(self, trigger=6, target=2.0, max_units=10):
        self.trigger = trigger
        self.target = target
        self.max_units = max_units
        self.reset()

    @property
    def name(self):
        return f"D'Alembert (T{self.trigger}, max{self.max_units}u)"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.units = 1

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.trigger:
                self.in_sequence = True
                self.units = 1
                return BetDecision(True, base_bet * self.units, self.target)
            return BetDecision(False)

        if multiplier >= self.target:  # WIN
            self.units = max(1, self.units - 1)
            if self.units == 1:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            return BetDecision(True, base_bet * self.units, self.target)
        else:  # LOSS
            self.units += 1
            if self.units > self.max_units:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            return BetDecision(True, base_bet * self.units, self.target)

In [ ]:
class LabouchereStrategy(BettingStrategy):
    """Labouchère: mantém uma lista de números.
    Aposta = soma do primeiro + último. Win: remove ambos. Loss: adiciona soma ao final."""

    def __init__(self, trigger=6, target=2.0, initial_list=None, max_list_size=12):
        self.trigger = trigger
        self.target = target
        self.initial_list = initial_list or [1, 2, 3]
        self.max_list_size = max_list_size
        self.reset()

    @property
    def name(self):
        return f"Labouchère (T{self.trigger})"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.number_list = []

    def _get_bet_units(self):
        if len(self.number_list) == 0:
            return 0
        if len(self.number_list) == 1:
            return self.number_list[0]
        return self.number_list[0] + self.number_list[-1]

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.trigger:
                self.in_sequence = True
                self.number_list = self.initial_list.copy()
                units = self._get_bet_units()
                return BetDecision(True, base_bet * units, self.target)
            return BetDecision(False)

        if multiplier >= self.target:  # WIN
            if len(self.number_list) <= 2:
                self.number_list = []
            else:
                self.number_list = self.number_list[1:-1]

            if len(self.number_list) == 0:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            units = self._get_bet_units()
            return BetDecision(True, base_bet * units, self.target)
        else:  # LOSS
            lost_units = self._get_bet_units()
            self.number_list.append(lost_units)
            if len(self.number_list) > self.max_list_size:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            units = self._get_bet_units()
            return BetDecision(True, base_bet * units, self.target)

In [ ]:
class OscarsGrindStrategy(BettingStrategy):
    """Oscar's Grind: aumenta 1 unidade APENAS após ganho.
    Meta: lucrar exatamente 1 unidade por ciclo. Muito conservador."""

    def __init__(self, trigger=6, target=2.0, max_units=6):
        self.trigger = trigger
        self.target = target
        self.max_units = max_units
        self.reset()

    @property
    def name(self):
        return f"Oscar's Grind (T{self.trigger})"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.units = 1
        self.cycle_profit = 0  # Em unidades

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.trigger:
                self.in_sequence = True
                self.units = 1
                self.cycle_profit = 0
                return BetDecision(True, base_bet, self.target)
            return BetDecision(False)

        if multiplier >= self.target:  # WIN
            self.cycle_profit += self.units * (self.target - 1)
            if self.cycle_profit >= 1:
                # Ciclo completo - lucrou 1+ unidade
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            self.units = min(self.units + 1, self.max_units)
            return BetDecision(True, base_bet * self.units, self.target)
        else:  # LOSS
            self.cycle_profit -= self.units
            if abs(self.cycle_profit) > self.max_units * 3:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            # Não aumenta após perda (diferença do Martingale)
            return BetDecision(True, base_bet * self.units, self.target)

In [ ]:
class KellyCriterionStrategy(BettingStrategy):
    """Kelly Criterion: aposta = (p*b - q) / b onde:
    p = prob. de ganho, q = prob. de perda, b = odds (target - 1).
    Usa estimativa rolling da prob. real."""

    def __init__(self, trigger=6, target=2.0, window=500, fraction=0.5):
        self.trigger = trigger
        self.target = target
        self.window = window
        self.fraction = fraction  # Fração do Kelly (0.5 = half-Kelly)
        self.reset()

    @property
    def name(self):
        return f"Kelly {self.fraction:.0%} (T{self.trigger}, w{self.window})"

    def reset(self):
        self.consecutive_lows = 0
        self.history = []

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD
        self.history.append(multiplier)

        if is_low:
            self.consecutive_lows += 1
        else:
            self.consecutive_lows = 0

        if self.consecutive_lows >= self.trigger:
            self.consecutive_lows = 0  # Reseta para não apostar toda rodada

            # Calcular Kelly baseado na prob. real recente
            recent = self.history[-self.window:] if len(self.history) >= self.window else self.history
            p_win = np.mean(np.array(recent) >= self.target)
            b = self.target - 1  # Odds
            q = 1 - p_win

            kelly_pct = (p_win * b - q) / b if b > 0 else 0
            kelly_pct = max(0, kelly_pct) * self.fraction

            if kelly_pct > 0:
                bet = bankroll * kelly_pct
                return BetDecision(True, bet, self.target)

        return BetDecision(False)

In [ ]:
class FixedFractionStrategy(BettingStrategy):
    """Aposta fixa: sempre X% da banca atual.
    Sem progressão, sem martingale. Puramente gestão de risco."""

    def __init__(self, trigger=6, target=2.0, fraction=0.02):
        self.trigger = trigger
        self.target = target
        self.fraction = fraction
        self.reset()

    @property
    def name(self):
        return f"Fixed {self.fraction:.1%} (T{self.trigger})"

    def reset(self):
        self.consecutive_lows = 0

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if is_low:
            self.consecutive_lows += 1
        else:
            self.consecutive_lows = 0

        if self.consecutive_lows >= self.trigger:
            self.consecutive_lows = 0
            bet = bankroll * self.fraction
            return BetDecision(True, bet, self.target)

        return BetDecision(False)

In [ ]:
class AdaptiveTriggerStrategy(BettingStrategy):
    """Trigger adaptativo: ajusta o trigger baseado na volatilidade recente.
    Alta volatilidade (muitos LOWs) → trigger mais alto.
    Baixa volatilidade → trigger mais baixo."""

    def __init__(self, base_trigger=6, target=2.0, window=200,
                 min_trigger=4, max_trigger=10, pattern=None):
        self.base_trigger = base_trigger
        self.target = target
        self.window = window
        self.min_trigger = min_trigger
        self.max_trigger = max_trigger
        self.pattern = pattern or [1, 2, 4]
        self.reset()

    @property
    def name(self):
        return f"Adaptativo (base T{self.base_trigger}, w{self.window})"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.current_dobra = 0
        self.low_history = []
        self.current_trigger = self.base_trigger

    def _update_trigger(self):
        if len(self.low_history) < self.window:
            return
        recent = self.low_history[-self.window:]
        pct_low = np.mean(recent)

        # Se mais LOWs que o normal → jogar mais conservador
        if pct_low > 0.58:
            self.current_trigger = min(self.base_trigger + 2, self.max_trigger)
        elif pct_low > 0.55:
            self.current_trigger = self.base_trigger + 1
        elif pct_low < 0.50:
            self.current_trigger = max(self.base_trigger - 1, self.min_trigger)
        else:
            self.current_trigger = self.base_trigger

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD
        self.low_history.append(1 if is_low else 0)
        self._update_trigger()

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0

            if self.consecutive_lows >= self.current_trigger:
                self.in_sequence = True
                self.current_dobra = 0
                mult = self.pattern[0]
                return BetDecision(True, base_bet * mult, self.target)
            return BetDecision(False)

        if multiplier >= self.target:  # WIN
            self.in_sequence = False
            self.consecutive_lows = 0
            return BetDecision(False)
        else:  # LOSS
            self.current_dobra += 1
            if self.current_dobra >= len(self.pattern):
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            mult = self.pattern[self.current_dobra]
            return BetDecision(True, base_bet * mult, self.target)

## 2. Torneio Principal (Target 2.0x)

In [ ]:
config = BankrollConfig(
    initial_bankroll=1000.0,
    base_bet_pct=0.0167,  # banca/6
    compound=False,
    stop_gain_pct=10.0,   # Sem stop gain para comparar livre
    stop_loss_pct=1.0,    # 100% stop loss
)

strategies = [
    MartingaleStrategy(trigger=6, target=2.0, pattern=[1, 2, 4]),
    MartingaleStrategy(trigger=6, target=2.0, pattern=[1, 2, 1, 2]),
    AntiMartingaleStrategy(trigger=6, target=2.0, max_wins=3),
    FibonacciStrategy(trigger=6, target=2.0, max_level=8),
    DAlembertStrategy(trigger=6, target=2.0, max_units=10),
    LabouchereStrategy(trigger=6, target=2.0),
    OscarsGrindStrategy(trigger=6, target=2.0),
    KellyCriterionStrategy(trigger=6, target=2.0, fraction=0.5),
    FixedFractionStrategy(trigger=6, target=2.0, fraction=0.02),
    AdaptiveTriggerStrategy(base_trigger=6, target=2.0, pattern=[1, 2, 4]),
]

results = []
for strat in strategies:
    print(f"Testando: {strat.name}...", end=" ")
    r = backtest(strat, multipliers, config)
    results.append(r)
    print(f"P&L: R${r.total_profit:+.0f} | WR: {r.win_rate:.0f}% | DD: {r.max_drawdown_pct:.1f}%")

print(f"\nTorneio completo: {len(results)} estratégias testadas")

In [ ]:
# Comparação visual
plot_comparison(results, 'Torneio de Estratégias - Target 2.0x')

## 3. Variação de Targets (1.5x, 2.0x, 2.5x, 3.0x)

In [ ]:
targets = [1.5, 2.0, 2.5, 3.0]
target_results = {}

for target in targets:
    print(f"\n=== Target {target}x ===")
    strats = [
        MartingaleStrategy(trigger=6, target=target, pattern=[1, 2, 4]),
        FibonacciStrategy(trigger=6, target=target),
        DAlembertStrategy(trigger=6, target=target),
        KellyCriterionStrategy(trigger=6, target=target, fraction=0.5),
        AdaptiveTriggerStrategy(base_trigger=6, target=target, pattern=[1, 2, 4]),
    ]

    target_results[target] = []
    for strat in strats:
        r = backtest(strat, multipliers, config)
        target_results[target].append(r)
        print(f"  {strat.name:40s} | P&L: R${r.total_profit:+8.0f} | Sharpe: {r.sharpe_ratio:+.3f}")

In [ ]:
# Heatmap: Lucro por Estratégia x Target
fig, ax = plt.subplots(figsize=(12, 6))

strat_names = [r.strategy_name for r in target_results[2.0]]
data = []
for target in targets:
    row = [r.total_profit for r in target_results[target]]
    data.append(row)

data = np.array(data)
im = ax.imshow(data, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(strat_names)))
ax.set_xticklabels([s.split('(')[0].strip() for s in strat_names], rotation=45, ha='right')
ax.set_yticks(range(len(targets)))
ax.set_yticklabels([f'{t}x' for t in targets])
ax.set_title('Lucro Total (R$) por Estratégia e Target')

for i in range(len(targets)):
    for j in range(len(strat_names)):
        ax.text(j, i, f'R${data[i,j]:+.0f}', ha='center', va='center',
                fontsize=9, color='black' if abs(data[i,j]) < data.max()/2 else 'white')

plt.colorbar(im, ax=ax, label='Lucro (R$)')
plt.tight_layout()
plt.show()

## 4. Variação de Triggers (4, 5, 6, 7, 8)

In [ ]:
triggers = [4, 5, 6, 7, 8]

# Testar martingale 1/2/4 com diferentes triggers
trigger_results = []
for t in triggers:
    strat = MartingaleStrategy(trigger=t, target=2.0, pattern=[1, 2, 4])
    r = backtest(strat, multipliers, config)
    r.strategy_name = f"Martingale T{t}"
    trigger_results.append(r)
    print(f"T{t}: P&L=R${r.total_profit:+.0f} | Apostas={r.total_bets} | WR={r.win_rate:.0f}% | DD={r.max_drawdown_pct:.1f}%")

plot_comparison(trigger_results, 'Martingale 1/2/4 - Variação do Trigger')

## 5. Ranking Final

In [ ]:
# Ranking por Sharpe Ratio (melhor relação risco/retorno)
all_results = sorted(results, key=lambda r: r.sharpe_ratio, reverse=True)

print("=" * 80)
print("RANKING POR SHARPE RATIO (risco ajustado)")
print("=" * 80)
print(f"{'#':>3} {'Estratégia':40s} {'Sharpe':>8} {'Lucro':>10} {'WR':>6} {'DD%':>7} {'PF':>6}")
print("-" * 80)

for i, r in enumerate(all_results, 1):
    print(f"{i:>3} {r.strategy_name:40s} {r.sharpe_ratio:>+8.3f} "
          f"R${r.total_profit:>+8.0f} {r.win_rate:>5.0f}% {r.max_drawdown_pct:>6.1f}% "
          f"{r.profit_factor:>5.2f}")